# Building patient features

Move the single-patient logic into reusable functions.

In [ ]:
import pandas as pd

base_url = "https://physionet.org/files/challenge-2019/1.0.0/training/training_setA/"

## Load a patient

In [ ]:
def load_patient(patient_id):
    url = base_url + patient_id + ".psv"
    patient = pd.read_csv(url, sep="|")
    return patient

The loader takes a patient ID and returns the corresponding DataFrame.

In [ ]:
patient = load_patient("p000001")
patient.head()

## Build one feature row

Current rule: use the first 6 observations as input and exclude patients already positive in that window.

In [ ]:
def build_patient_features(patient_id):
    patient = load_patient(patient_id)

    if len(patient) <= 6:
        return None

    first_6h = patient.head(6)
    future = patient.iloc[6:]

    if first_6h["SepsisLabel"].max() == 1:
        return None

    valid_hr = first_6h["HR"].dropna()

    if len(valid_hr) >= 2:
        hr_change = valid_hr.iloc[-1] - valid_hr.iloc[0]
    else:
        hr_change = float("nan")

    return {
        "patient": patient_id,
        "mean_HR_6h": first_6h["HR"].mean(),
        "min_HR_6h": first_6h["HR"].min(),
        "max_HR_6h": first_6h["HR"].max(),
        "std_HR_6h": first_6h["HR"].std(),
        "change_HR_6h": hr_change,
        "min_O2Sat_6h": first_6h["O2Sat"].min(),
        "mean_Resp_6h": first_6h["Resp"].mean(),
        "mean_MAP_6h": first_6h["MAP"].mean(),
        "future_sepsis": future["SepsisLabel"].max()
    }

`None` means the patient does not fit the current setup, for example because the stay is too short or the label is already positive.

## Test on one patient

In [ ]:
result = build_patient_features("p000001")
result

The output is one dictionary that can become one row in a modelling dataset.

## Early-positive example

In [ ]:
build_patient_features("p000034")

## Apply it to multiple patients

In [ ]:
dataset_rows = []

for i in range(1, 101):
    patient_id = f"p{i:06d}"
    row = build_patient_features(patient_id)

    if row is not None:
        dataset_rows.append(row)

dataset = pd.DataFrame(dataset_rows)
dataset

At this stage: one row = one eligible patient, engineered columns = features, `future_sepsis` = target.

In [ ]:
print("Eligible patients:", len(dataset))
dataset["future_sepsis"].value_counts(dropna=False)

## Missing values after feature engineering

In [ ]:
dataset.isna().sum().sort_values(ascending=False)

This is still a temporary setup. The next version should use a fixed future horizon rather than 'any later positive label'.